In [2]:
import pandas as pd
import numpy as np
root="/home/jacoponudo/Documents/Virality-on-Shorts/" # TO CHANGE according to your local path

# Carichiamo i dataset puliti
instagram = pd.read_csv(root + "data/clean/instagram.csv")
tiktok = pd.read_csv(root + "data/clean/tiktok.csv")
youtube = pd.read_csv(root + "data/clean/youtube.csv")

# Filtriamo i publisher attivi per ogni piattaforma
from tools import filter_active_publisher_years
youtube_active, yt_publishers, yt_counts = filter_active_publisher_years(
    df=youtube,
    platform_name="YouTube",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=70
)

instagram_active, ig_publishers, ig_counts = filter_active_publisher_years(
    df=instagram,
    platform_name="Instagram",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=70
)

tiktok_active, tt_publishers, tt_counts = filter_active_publisher_years(
    df=tiktok,
    platform_name="TikTok",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=70
)


# Selezioniamo le colonne di interesse
tiktok=tiktok_active[['video_id','publisher', 'year','views','platform']]
youtube=youtube_active[['video_id','publisher', 'year','views','platform']]
instagram=instagram_active[['video_id','publisher', 'year','views','platform']]

tiktok['views'] =np.log(tiktok['views'].astype(int) + 1)
youtube['views'] =np.log(youtube['views'].astype(int) + 1)
instagram['views'] =np.log(instagram['views'].astype(int) + 1)


YouTube
-------
Original videos:              51,863
Original publishers:          130
Videos after filtering:       47,185
Unique publishers retained:   53

Publisher-years retained by year:
year
2022     8
2023    17
2024    24
2025    42
2026    39
Name: count, dtype: int64

Instagram
---------
Original videos:              235,743
Original publishers:          161
Videos after filtering:       227,976
Unique publishers retained:   130

Publisher-years retained by year:
year
2022     30
2023     97
2024    103
2025    121
2026    103
Name: count, dtype: int64

TikTok
------
Original videos:              195,672
Original publishers:          160
Videos after filtering:       189,948
Unique publishers retained:   119

Publisher-years retained by year:
year
2022    28
2023    60
2024    76
2025    94
2026    86
Name: count, dtype: int64


In [3]:
"""
Option F - Shannon Entropy (diversity/dispersion of virality)
==========================================================================

Metric: normalized Shannon entropy of the (log-)views distribution,
computed per channel-year.

Rationale: while b-value and quantile-ratio capture the "heaviness"
of the upper tail, entropy captures how *spread out* / unpredictable
the whole distribution of views is for a channel. A channel where
every video gets roughly the same number of views has LOW entropy
(concentrated/predictable distribution). A channel where views are
scattered across many different orders of magnitude (a few huge
hits, many flops) has HIGH entropy (dispersed/unpredictable
distribution).

Estimator
---------
Views (already natural-log transformed) are discretized into N_BINS
equal-width bins. Shannon entropy is computed on the resulting
histogram probabilities:
    H = - sum( p_i * log(p_i) )   for p_i > 0
and then normalized by log(N_BINS) so that H_norm in [0, 1]:
    H_norm = 1  -> maximum dispersion (uniform over bins)
    H_norm = 0  -> maximum concentration (all mass in one bin)
Normalizing makes entropy comparable across channel-years even if
N_BINS is changed.

Bootstrap rationale
--------------------
Entropy estimates from histograms are sensitive to sample size (few
videos = noisy/unstable histogram). We resample with replacement to
a fixed N=100 per channel-year, repeated B=100 times, and average, so
channels with different activity levels are compared at equal
exposure.

Expected input
--------------
tiktok, youtube, instagram: DataFrames with columns
  ['video_id', 'publisher', 'year', 'views', 'platform']
where 'views' is already np.log(views + 1).

Plots (for paper)
-----------------
Fig. 1: Boxplot of entropy_mean by platform -> distribution comparison
Fig. 2: entropy_mean vs n_videos_original by platform -> confound check
Fig. 3: entropy_mean trend over year by platform (mean +/- 95% CI)
"""

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

np.random.seed(42)

N_BOOT = 100
B_REPLICATES = 100

# --- Parametro iniziale: numero di bin usati per discretizzare la
#     distribuzione delle views e stimare l'entropia di Shannon ---
N_BINS = 10

FIGURE_DIR = "figures_entropy"

# ----------------------------------------------------------------------
# Publication-style matplotlib settings
# ----------------------------------------------------------------------
PLATFORM_COLORS = {
    "TikTok": "#1f77b4",
    "YouTube": "#d62728",
    "Instagram": "#9467bd",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "legend.frameon": False,
})


def shannon_entropy(x, bins=N_BINS, normalize=True):
    """Normalized Shannon entropy of x, estimated via equal-width histogram.

    x: array of 'magnitudes' (here: natural-log views).
    bins: number of equal-width bins used to discretize x.
    normalize: if True, divide by log(bins) so the result is in [0, 1].
    """
    x = np.asarray(x)
    if len(x) < 2 or np.allclose(x.min(), x.max()):
        return np.nan

    counts, _ = np.histogram(x, bins=bins)
    counts = counts[counts > 0]
    if len(counts) == 0:
        return np.nan

    p = counts / counts.sum()
    H = -np.sum(p * np.log(p))

    if normalize:
        H = H / np.log(bins)
    return H


def bootstrap_entropy(all_data, bins=N_BINS):
    records = []
    for key, grp in all_data.groupby("group_key"):
        views = grp["log_views"].values
        platform = grp["platform"].iloc[0]
        publisher = grp["publisher"].iloc[0]
        year = grp["year"].iloc[0]
        n_original = len(views)

        entropy_boot = []
        for _ in range(B_REPLICATES):
            sample = np.random.choice(views, size=N_BOOT, replace=True)
            entropy_boot.append(shannon_entropy(sample, bins=bins))

        records.append({
            "platform": platform,
            "publisher": publisher,
            "year": year,
            "n_videos_original": n_original,
            "entropy_mean": np.nanmean(entropy_boot),
            "entropy_ci_low": np.nanpercentile(entropy_boot, 2.5),
            "entropy_ci_high": np.nanpercentile(entropy_boot, 97.5),
        })
    return pd.DataFrame(records)


def _platform_order(summary):
    """Consistent platform ordering/colors across all figures."""
    preferred = [p for p in ["TikTok", "YouTube", "Instagram"] if p in summary["platform"].unique()]
    return preferred or sorted(summary["platform"].unique())


def _color_for(platform):
    return PLATFORM_COLORS.get(platform, "#7f7f7f")


def plot_boxplot_by_platform(summary, out_dir):
    """Fig. 1: distribution of entropy_mean by platform (boxplot + jittered points)."""
    platforms = _platform_order(summary)
    fig, ax = plt.subplots(figsize=(6, 4.5))

    data = [summary.loc[summary["platform"] == p, "entropy_mean"].dropna().values for p in platforms]
    bp = ax.boxplot(
        data,
        widths=0.5,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 1.5},
    )
    # Set tick labels manually (avoids the 'labels'/'tick_labels' kwarg
    # rename across matplotlib versions: removed in 3.9+, renamed in 3.11+)
    ax.set_xticks(range(1, len(platforms) + 1))
    ax.set_xticklabels(platforms)
    for patch, p in zip(bp["boxes"], platforms):
        patch.set_facecolor(_color_for(p))
        patch.set_alpha(0.35)
        patch.set_edgecolor(_color_for(p))

    rng = np.random.default_rng(0)
    for i, (p, vals) in enumerate(zip(platforms, data), start=1):
        jitter = rng.normal(0, 0.06, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals,
                   color=_color_for(p), alpha=0.5, s=14, linewidths=0)

    ax.set_ylabel("Normalized Shannon entropy of views")
    ax.set_xlabel("Platform")
    ax.set_title("Dispersion of virality by platform\n(channel-year level, bootstrapped)")

    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(out_dir, f"fig1_entropy_boxplot_by_platform.{ext}"))
    plt.close(fig)


def plot_entropy_vs_activity(summary, out_dir):
    """Fig. 2: entropy_mean vs channel activity (n_videos_original), by platform."""
    platforms = _platform_order(summary)
    fig, ax = plt.subplots(figsize=(6, 4.5))

    for p in platforms:
        sub = summary[summary["platform"] == p].dropna(subset=["entropy_mean"])
        ax.scatter(
            sub["n_videos_original"], sub["entropy_mean"],
            color=_color_for(p), alpha=0.55, s=20, label=p, linewidths=0,
        )
        if len(sub) > 2:
            coeffs = np.polyfit(sub["n_videos_original"], sub["entropy_mean"], 1)
            x_line = np.linspace(sub["n_videos_original"].min(), sub["n_videos_original"].max(), 50)
            ax.plot(x_line, np.polyval(coeffs, x_line), color=_color_for(p), linewidth=1.8)

    ax.set_xlabel("Channel activity (videos published per year)")
    ax.set_ylabel("Normalized Shannon entropy of views")
    ax.set_title("Views entropy vs. channel activity\n(checking for activity confound)")
    ax.legend(title="Platform", loc="best")

    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(out_dir, f"fig2_entropy_vs_activity.{ext}"))
    plt.close(fig)


def plot_entropy_trend_by_year(summary, out_dir):
    """Fig. 3: yearly mean entropy_mean +/- 95% CI, by platform."""
    platforms = _platform_order(summary)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))

    for p in platforms:
        sub = summary[summary["platform"] == p].dropna(subset=["entropy_mean"]).copy()
        sub["year"] = sub["year"].astype(int)
        grouped = sub.groupby("year")["entropy_mean"].agg(["mean", "std", "count"]).reset_index()
        grouped["se"] = grouped["std"] / np.sqrt(grouped["count"])
        grouped["ci95"] = 1.96 * grouped["se"]
        grouped = grouped.sort_values("year")

        ax.plot(grouped["year"], grouped["mean"], marker="o", color=_color_for(p), label=p)
        ax.fill_between(
            grouped["year"],
            grouped["mean"] - grouped["ci95"],
            grouped["mean"] + grouped["ci95"],
            color=_color_for(p), alpha=0.18, linewidth=0,
        )

    ax.set_xlabel("Year")
    ax.set_ylabel("Mean normalized Shannon entropy of views")
    ax.set_title("Trend of views-entropy over time, by platform")
    ax.set_xticks(sorted(summary["year"].astype(int).unique()))
    ax.legend(title="Platform", loc="best")

    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(out_dir, f"fig3_entropy_trend_by_year.{ext}"))
    plt.close(fig)


def generate_plots(summary, out_dir=FIGURE_DIR):
    os.makedirs(out_dir, exist_ok=True)
    plot_boxplot_by_platform(summary, out_dir)
    plot_entropy_vs_activity(summary, out_dir)
    plot_entropy_trend_by_year(summary, out_dir)
    print(f"\nSaved 3 figures (PDF + PNG) to '{out_dir}/'")


def main(bins=N_BINS):

    all_data = pd.concat([tiktok, youtube, instagram], ignore_index=True)
    all_data = all_data.rename(columns={"views": "log_views"})
    all_data["group_key"] = (
        all_data["platform"].astype(str) + "_" +
        all_data["publisher"].astype(str) + "_" +
        all_data["year"].astype(str)
    )

    # --- 2. Bootstrap the entropy metric per channel-year ---
    summary = bootstrap_entropy(all_data, bins=bins)
    summary["year"] = summary["year"].astype(str)

    print("Preview of channel-year summary table:")
    print(summary.head())
    print(f"\nTotal channel-years analyzed: {len(summary)}")
    print(f"Numero di bin usato per l'entropia: {bins}")

    # --- 3. Mixed model: entropy ~ platform + year + activity, random effect per channel ---
    model_F = smf.mixedlm(
        "entropy_mean ~ platform + year + n_videos_original",
        data=summary,
        groups=summary["publisher"],
    ).fit()

    print("\n" + "=" * 70)
    print("OPTION F - Shannon Entropy of views (normalized, histogram-based)")
    print("=" * 70)
    print(model_F.summary())

    summary.to_csv("virality_option_f_summary.csv", index=False)
    print("\nSaved: virality_option_f_summary.csv")

    # --- 4. Plots for the paper ---
    generate_plots(summary)


if __name__ == "__main__":
    main(bins=N_BINS)

Preview of channel-year summary table:
    platform  publisher  year  n_videos_original  entropy_mean  \
0  Instagram      AMICA  2023                101      0.861617   
1  Instagram      AMICA  2024                189      0.768285   
2  Instagram      AMICA  2025                243      0.729060   
3  Instagram      AMICA  2026                103      0.828910   
4  Instagram  Adnkronos  2023                128      0.770588   

   entropy_ci_low  entropy_ci_high  
0        0.806146         0.936275  
1        0.655829         0.888063  
2        0.626473         0.820522  
3        0.776023         0.883106  
4        0.683666         0.929083  

Total channel-years analyzed: 928
Numero di bin usato per l'entropia: 10


/home/jacoponudo/Documents/Virality-on-Shorts/.venv/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



OPTION F - Shannon Entropy of views (normalized, histogram-based)
            Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  entropy_mean
No. Observations:   928      Method:              REML        
No. Groups:         285      Scale:               0.0038      
Min. group size:    1        Log-Likelihood:      1080.4846   
Max. group size:    10       Converged:           Yes         
Mean group size:    3.3                                       
--------------------------------------------------------------
                    Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept            0.834    0.009 88.115 0.000  0.816  0.853
platform[T.TikTok]   0.020    0.008  2.570 0.010  0.005  0.035
platform[T.YouTube] -0.007    0.009 -0.856 0.392 -0.024  0.009
year[T.2023]        -0.022    0.009 -2.300 0.021 -0.040 -0.003
year[T.2024]        -0.015    0.009 -1.641 0.101 -0.034  0.003
y